# Dataset Construction for Preference Alignment (DPO)

This notebook prepares the **judge prompts** used for LLM-as-a-Judge inference.

This notebook constructs preference-style datasets required for Direct Preference Optimization (DPO).

Given an existing dataset containing chosen and rejected responses, we convert each example into a judge-ready prompt that allows an LLM to compare two candidate answers and select the better one.

This follows the standard preference learning paradigm used in RLHF and DPO pipelines (Rafailov et al., 2023).


In [ ]:
from pathlib import Path
from utils.dataset_helpers import (
    set_seed,
    load_parquet_dataset,
    build_judge_dataset,
    save_dataset,
    preview_samples,
)

try:
    REPO_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    REPO_ROOT = Path.cwd()
    
PARQUET_PATH = REPO_ROOT / "data" / "train_sponsor_filtered.parquet"
SAVE_DIR     = REPO_ROOT / "Sky_dataset"



### Reproducibility Control

Since answer ordering is randomly flipped during dataset construction, we fix the random seed to ensure deterministic generation of preference pairs.

Reproducibility is critical in alignment research to ensure that observed improvements arise from model updates rather than stochastic sampling artifacts.

In [ ]:
set_seed(2021)

### Loading Source Dataset

We load the source dataset in Parquet format. Each record contains a prompt and two responses: one preferred (chosen) and one non-preferred (rejected).

This format aligns with common reward-model and DPO datasets used in alignment research.

In [ ]:
raw_dataset = load_parquet_dataset(str(PARQUET_PATH))
print(raw_dataset)

### Judge Prompt Construction

To enable LLM-based comparison, we transform each example into a structured evaluation prompt.

For each question, we present two candidate answers and instruct the judge model to select the better one based on:

- Coherence
- Accuracy
- Coverage
- Overall quality

We randomly flip answer ordering to prevent positional bias and ensure robustness of preference supervision.

This LLM-as-a-judge approach follows recent evaluation paradigms:

- Zheng et al., 2023 — Judging LLMs by Their Covers (MT-Bench)

- Liu et al., 2023 — G-Eval: NLG Evaluation using GPT-4

In [ ]:
train_dataset = build_judge_dataset(raw_dataset)
print(train_dataset)

In [ ]:
preview_samples(train_dataset, n=3)

### Persisting Dataset

The constructed dataset is saved in HuggingFace Dataset format to ensure compatibility with downstream DPO training and evaluation pipelines.

This structured storage enables reproducible experimentation and efficient loading during training.

In [ ]:
final_dataset = save_dataset(train_dataset, SAVE_DIR)
print("Dataset saved to:", SAVE_DIR)


References

- Rafailov et al., 2023
Direct Preference Optimization: Your Language Model is Secretly a Reward Model
https://arxiv.org/abs/2305.18290

- Zheng et al., 2023
Judging LLMs by Their Covers: A Benchmark for Open-ended Question Answering (MT-Bench & Chatbot Arena)
https://arxiv.org/abs/2306.05685